# Setup smoke test

Verify Tidy3D installation, authentication, and a minimal straight-waveguide FDTD job on the cloud.

**Prerequisites:** `pip install -e ".[dev]"` and `tidy3d configure`.

In [ ]:
import tidy3d as td
import tidy3d.web as web
import matplotlib.pyplot as plt

from fdtd_pic.config import FREQ0, FWIDTH, HEIGHT, WAVELENGTH
from fdtd_pic.materials import si_medium, sio2_medium

print('Tidy3D', td.__version__)
acct = web.account(verbose=False)
print('Account:', acct)

In [ ]:
# Minimal straight waveguide FDTD (docs-style smoke test)
width = 0.5
sim_size = (5.0, 3.0, HEIGHT + 1.0)
wg = td.Structure(
    geometry=td.Box(center=(0, 0, 0), size=(3.0, width, HEIGHT)),
    medium=si_medium(),
)
sim = td.Simulation(
    size=sim_size,
    structures=[wg],
    sources=[
        td.ModeSource(
            center=(-1.2, 0, 0),
            size=(0, 2.0, HEIGHT + 1.0),
            source_time=td.GaussianPulse(freq0=FREQ0, fwidth=FWIDTH),
            direction='+',
            mode_spec=td.ModeSpec(num_modes=1),
        )
    ],
    monitors=[
        td.FieldMonitor(center=(0, 0, 0), size=(td.inf, td.inf, 0), freqs=[FREQ0], name='field')
    ],
    grid_spec=td.GridSpec.auto(min_steps_per_wvl=15, wavelength=WAVELENGTH),
    run_time=1e-12,
    boundary_spec=td.BoundarySpec.all_sides(boundary=td.PML()),
    medium=sio2_medium(),
)
sim.plot(z=0)
plt.show()

In [ ]:
# Submit to cloud (uncomment when authenticated)
# sim_data = web.run(sim, task_name='smoke_straight_wg', path='smoke_straight_wg.hdf5', verbose=True)
# sim_data.plot_field('field', 'E', 'abs^2')
# plt.show()

## Key takeaway

If credits print and the geometry plot looks correct, the pipeline is ready for Modules 1–3. Local mode solving (no cloud) is available via `python scripts/smoke_test.py`.